# 02 - Target Architecture: The Blueprint

**Goal:** Define the target architecture for a properly MLOps-integrated spine segmentation system. This is the blueprint for the real integration work.

---

## High-Level Architecture

```
┌─────────────────────────────────────────────────────────────────────────────────┐
│                              AZURE RESOURCE GROUP                               │
│                                                                                 │
│  ┌──────────────────┐  ┌──────────────────┐  ┌───────────────────────────────┐  │
│  │  Azure Blob      │  │  Azure Container  │  │  MLflow Tracking Server       │  │
│  │  Storage          │  │  Registry (ACR)   │  │  (VM or Azure ML Workspace)   │  │
│  │                  │  │                  │  │                               │  │
│  │  • Raw CT scans  │  │  • Inference     │  │  • Experiments                │  │
│  │  • Annotations   │  │    Docker images │  │  • Run metrics/params         │  │
│  │  • DVC cache     │  │  • SegRefine     │  │  • Model Registry             │  │
│  │                  │  │    base image    │  │  • Artifacts (configs, plots)  │  │
│  └────────┬─────────┘  └────────┬─────────┘  └──────────────┬────────────────┘  │
│           │                     │                            │                   │
└───────────┼─────────────────────┼────────────────────────────┼───────────────────┘
            │                     │                            │
            │ dvc push/pull       │ docker push               │ mlflow log/query
            │                     │                            │
┌───────────┼─────────────────────┼────────────────────────────┼───────────────────┐
│           │              GitLab / GitHub                     │                   │
│           │                                                  │                   │
│  ┌────────┴─────────┐  ┌──────────────────┐  ┌──────────────┴────────────────┐  │
│  │ maia-data-       │  │ maia-spine-       │  │ maia-spine-                   │  │
│  │ registry         │  │ training          │  │ inference                     │  │
│  │                  │  │                  │  │                               │  │
│  │ • .dvc pointers  │  │ • Training code  │  │ • Inference code              │  │
│  │ • Dataset docs   │  │ • DVC pipeline   │  │ • Dockerfile                  │  │
│  │ • Split CSVs     │  │ • MLflow logging │  │ • CI/CD → ACR                 │  │
│  └──────────────────┘  └──────────────────┘  └───────────────────────────────┘  │
│                                                                                 │
│                        ┌──────────────────┐                                     │
│                        │ maia-ml-core      │                                     │
│                        │ (shared package)  │                                     │
│                        │ • config.py       │                                     │
│                        │ • classes.py      │                                     │
│                        │ • transforms/     │                                     │
│                        │ • dataset.py      │                                     │
│                        │ • model factory   │                                     │
│                        └──────────────────┘                                     │
└─────────────────────────────────────────────────────────────────────────────────┘
```

## Repo Strategy: 4 Repositories

| Repo | Purpose | Contents |
|------|---------|----------|
| **maia-data-registry** | Dataset catalog | `.dvc` pointers, split CSVs, dataset documentation |
| **maia-spine-training** | Training pipeline | Training scripts, DVC pipeline, MLflow integration |
| **maia-spine-inference** | Inference service | Inference engine, Dockerfile, CI/CD to ACR |
| **maia-ml-core** | Shared code | Config, classes, transforms, dataset builders, model factory |

### Why Not a Mono-Repo?

| Mono-repo | Multi-repo (recommended) |
|-----------|-------------------------|
| Simpler git history | Clear ownership per component |
| Easier cross-cutting changes | Independent CI/CD pipelines |
| One place for everything | Training CI doesn't trigger inference build |
| Gets messy at scale | Data registry independent of code |
| Docker build context includes everything | Smaller, focused Docker contexts |

Multi-repo works better because:
1. **Data** changes independently of **code**
2. **Training** runs are independent of **inference** releases
3. **Shared code** is versioned as a package (like any dependency)
4. Different team members work on different repos

## Shared Code Package: `maia-ml-core`

Extract the ~70% duplicated code into a Python package:

```
maia-ml-core/
├── pyproject.toml                   # Package metadata, version
├── src/
│   └── maia_ml_core/
│       ├── __init__.py
│       ├── config.py                # Unified config loading
│       ├── classes.py               # All vertebra/anatomy class definitions
│       ├── dataset.py               # DataLoader factories
│       ├── model.py                 # SwinUNETR model factory
│       └── transform/
│           ├── __init__.py
│           ├── custom.py            # Custom MONAI transforms
│           ├── utils.py
│           └── composition/
│               ├── __init__.py
│               ├── semantic.py      # Region task transforms
│               └── instance.py      # Instance task transforms
└── tests/
    ├── test_config.py
    ├── test_classes.py
    └── test_transforms.py
```

Usage in training and inference repos:

```python
# requirements.txt (in training or inference repo)
maia-ml-core @ git+https://gitlab.com/maia/maia-ml-core.git@v1.2.0

# In code
from maia_ml_core.config import load_config
from maia_ml_core.classes import get_model_classes
from maia_ml_core.model import create_swinunetr
from maia_ml_core.transform.composition.semantic import build_transforms
```

**Benefit:** Fix a bug once, both repos get it on next version bump.

## Data Flow: End-to-End

```
Step 1: DATA ARRIVES
─────────────────────────────────────────────────────
NAS (current)  ──migration──→  Azure Blob Storage
                               ├── raw/spine/batch_2025_01/
                               ├── raw/spine/batch_2025_06/
                               └── raw/knee/batch_2025_03/

Step 2: REGISTER IN CATALOG
─────────────────────────────────────────────────────
maia-data-registry repo:
  $ dvc add raw/spine/batch_2025_06   → creates .dvc pointer
  $ git commit -m "Add 200 new spine CTs (June 2025)"
  $ dvc push                          → data stays on Azure Blob

Step 3: CREATE SPLITS
─────────────────────────────────────────────────────
maia-data-registry repo:
  splits/
  └── spine_semantic_v8/
      ├── train.csv                   # 320 patients
      ├── val.csv                     # 80 patients
      └── metadata.json               # split rationale
  $ dvc add splits/spine_semantic_v8
  $ git commit -m "Create train/val split for semantic v8"

Step 4: TRAINING
─────────────────────────────────────────────────────
maia-spine-training repo:
  $ dvc import maia-data-registry raw/spine  → pulls data
  $ dvc repro                                → runs pipeline
      prepare_data → train → evaluate
  MLflow logs: params, metrics, model, config

Step 5: MODEL PROMOTION
─────────────────────────────────────────────────────
MLflow Registry:
  spine-semantic-model v9 → Staging → validate → Production

Step 6: INFERENCE RELEASE
─────────────────────────────────────────────────────
maia-spine-inference repo:
  - Pull model from MLflow Registry (not bundled)
  - docker build → ACR push
  - Docker image tagged with model version
```

## Training Repo Architecture

```
maia-spine-training/
├── dvc.yaml                         # Pipeline definition
├── dvc.lock                         # Pinned versions of everything
├── params.yaml                      # Shared pipeline parameters
│
├── configs/
│   ├── spine_semantic.yaml          # Region model config
│   └── spine_instance.yaml          # Instance model config
│
├── src/
│   ├── prepare_data.py              # Data preparation stage
│   ├── train_semantic.py            # Region model training
│   ├── train_instance.py            # Instance model training
│   ├── evaluate.py                  # Model evaluation
│   └── register_model.py            # MLflow model registration
│
├── tests/
│   └── ...
│
├── requirements.txt                 # maia-ml-core + DVC + MLflow
├── .gitlab-ci.yml                   # lint → test → (manual train trigger)
└── README.md
```

### DVC Pipeline Definition

```yaml
# dvc.yaml
stages:

  prepare_data:
    cmd: python src/prepare_data.py
    deps:
      - src/prepare_data.py
      - data/raw/                    # DVC-tracked
    params:
      - data                         # from params.yaml
    outs:
      - data/prepared/

  train_semantic:
    cmd: python src/train_semantic.py --config configs/spine_semantic.yaml
    deps:
      - src/train_semantic.py
      - data/prepared/
      - configs/spine_semantic.yaml
    params:
      - model.semantic
      - training.semantic
    outs:
      - models/semantic/

  train_instance:
    cmd: python src/train_instance.py --config configs/spine_instance.yaml
    deps:
      - src/train_instance.py
      - data/prepared/
      - configs/spine_instance.yaml
    params:
      - model.instance
      - training.instance
    outs:
      - models/instance/

  evaluate:
    cmd: python src/evaluate.py
    deps:
      - src/evaluate.py
      - models/semantic/best_metric_model.pth
      - models/instance/best_metric_model.pth
      - data/prepared/
    metrics:
      - metrics/semantic.json:
          cache: false
      - metrics/instance.json:
          cache: false
```

### MLflow Integration Points

```python
# In train_semantic.py — minimal changes to existing code
import mlflow
from maia_ml_core.config import load_config

config = load_config("configs/spine_semantic.yaml")

with mlflow.start_run(run_name=f"{config['task']['name']}_{config['task']['version']}"):
    # Log all config as params
    mlflow.log_params(flatten_dict(config))  # flatten nested YAML
    mlflow.log_artifact("configs/spine_semantic.yaml")
    
    # Inside training loop (add 2 lines):
    mlflow.log_metric("train_loss", loss.item(), step=global_step)
    
    # After validation (add 2 lines):
    mlflow.log_metric("val_dice_mean", mean_dice, step=global_step)
    
    # Save model to registry:
    mlflow.pytorch.log_model(model, "model")
```

## Inference Repo Architecture

```
maia-spine-inference/
├── src/
│   ├── inference_pipeline.py        # 2-stage inference engine
│   ├── inferrer.py                  # Sequential inference
│   ├── main_pipeline.py             # Pipeline orchestration
│   └── docker_entrypoint.py         # CLI wrapper
│
├── configs/
│   ├── spine_semantic.yaml          # Inference-specific config
│   └── spine_instance.yaml
│
├── scripts/
│   └── download_model.py            # Pull model from MLflow Registry
│
├── Dockerfile
├── .gitlab-ci.yml                   # lint → test → build → scan → push
├── requirements.txt                 # maia-ml-core (no DVC/MLflow needed)
└── README.md
```

### Key Change: Model Loading

```python
# CURRENT: weights bundled in Docker image
model.load_state_dict(torch.load("/app/weight/training_.../best_metric_model.pth"))

# OPTION A: Pull from MLflow Registry at build time (recommended for production)
# In Dockerfile or build script:
#   python scripts/download_model.py --model spine-semantic --stage Production
# Then bundle the downloaded weights

# OPTION B: Pull from Azure Blob at runtime
# Mount blob storage or download at container start
# More flexible but requires network at inference time

# OPTION C: Pull from MLflow at runtime
# model = mlflow.pytorch.load_model("models:/spine-semantic-model/Production")
# Most flexible but adds MLflow dependency to inference
```

**Recommendation: Option A** — download at build time, bundle in image. Same as today, but the source is the MLflow Registry instead of a local folder. The Docker image stays self-contained (no network needed at inference).

## CI/CD Blueprint

### Training Repo CI/CD

```
maia-spine-training/.gitlab-ci.yml

On every push:
  ├── [quality] ruff lint + format
  ├── [test] pytest (unit tests for data prep, transforms)
  └── (stop here for branches)

On tag (v*):
  ├── [quality] ruff lint + format
  ├── [test] pytest
  └── [train] (manual trigger)
      ├── dvc pull (get data from Azure Blob)
      ├── dvc repro (run pipeline)
      ├── MLflow logs everything
      └── Model registered in MLflow Registry (Staging)

Model promotion (manual):
  └── Team validates → promotes to Production in MLflow UI
```

### Inference Repo CI/CD

```
maia-spine-inference/.gitlab-ci.yml

On every push:
  ├── [quality] ruff lint + format
  └── [test] pytest

On tag (v*):
  ├── [quality] ruff lint + format + pip-audit + SBOM
  ├── [build]
  │   ├── python scripts/download_model.py  (from MLflow: Production stage)
  │   ├── docker build (weights now in image)
  │   └── artifact: .tar.gz
  ├── [scan] Trivy security scan
  ├── [push] docker push → maiamvpacr.azurecr.io/verifine-spine/...:VERSION
  └── [cleanup]
```

### Shared Code CI/CD

```
maia-ml-core/.gitlab-ci.yml

On every push:
  ├── [quality] ruff lint + format
  └── [test] pytest (config loading, class mapping, transform shapes)

On tag (v*):
  └── [publish] Build + publish Python package (private PyPI or git tag)
```

## Azure Resource Topology

```
Azure Subscription (Maia startup credits)
│
└── Resource Group: maia-ml
    │
    ├── Storage Account: maiadata
    │   ├── Container: dvc-storage          ← DVC remote (all datasets)
    │   ├── Container: mlflow-artifacts     ← MLflow artifact store
    │   └── Container: raw-uploads          ← Landing zone for new data
    │
    ├── Container Registry: maiamvpacr
    │   ├── verifine-spine/segmentation/base:0.16.0     ← Inference images
    │   ├── verifine-spine/segmentation/base:0.17.0
    │   └── segrefine/base:latest                        ← SegRefine base
    │
    ├── Virtual Machine: mlflow-server (optional)
    │   └── MLflow tracking server + model registry
    │   └── Or: use Azure ML Workspace instead (managed)
    │
    └── Azure ML Workspace: maia-ml-workspace (optional)
        ├── MLflow tracking (built-in)
        ├── Model Registry (built-in)
        ├── Compute clusters (GPU training)
        └── Lineage graphs (FDA-ready)
```

### Cost Optimization

| Resource | Cost Strategy |
|----------|---------------|
| Blob Storage | Pay per GB stored + transfer. Hot tier for active data, Cool for archived |
| ACR | Basic tier is ~$5/month. Enough for a startup |
| MLflow VM | Small VM (~$30/month) or use Azure ML Workspace (pay per compute) |
| GPU compute | Use Azure ML clusters with min_instances=0 (scale to zero when idle) |

## Where Everything Lives

| What | Where | Why |
|------|-------|-----|
| **Python code** | Git (GitLab/GitHub) | Version control, code review |
| **Raw CT data** | Azure Blob Storage | Too large for Git |
| **Data pointers** | maia-data-registry (Git) | DVC `.dvc` files, tiny |
| **Train/val splits** | maia-data-registry (Git + DVC) | Tracked alongside data |
| **Training configs** | maia-spine-training (Git) | Versioned with training code |
| **Inference configs** | maia-spine-inference (Git) | Versioned with inference code |
| **Model weights** | MLflow Registry → Azure Blob | Versioned, traceable to run |
| **Docker images** | Azure Container Registry | Built by CI/CD |
| **Experiment logs** | MLflow server (or Azure ML) | Params, metrics, artifacts |
| **TensorBoard logs** | Alongside MLflow (keep both) | Detailed training curves |
| **SegRefine binary** | ACR base image | Proprietary, separate lifecycle |
| **SBOM / audit** | GitLab CI artifacts | 30-day retention |
| **Documentation** | Git repos (READMEs) | Close to code |

## Versioning Strategy

```
WHAT                    HOW                           EXAMPLE
────                    ───                           ───────
Shared package          Semantic versioning (git tag) maia-ml-core v1.2.0
Training code           Git tags                      maia-spine-training v0.9.0
Inference code          Git tags                      maia-spine-inference v0.17.0
Docker image            Git tag → Docker tag          verifine-spine:0.17.0
Dataset                 DVC hash + git commit         data/spine.dvc (hash: ab12cd34)
Train/val split         DVC hash + git commit         splits/v8.dvc
Trained model           MLflow auto-version           spine-semantic-model v9
Config                  Part of git tag               Committed with training code
SegRefine               ACR tag                       segrefine/base:1.0.0
```

### Version Dependency Chain

```
Docker image v0.17.0
  ├── uses model: spine-semantic v9 (from MLflow)
  │     └── trained with: maia-spine-training v0.9.0
  │           ├── config: spine_semantic.yaml (git commit abc123)
  │           ├── data: spine.dvc hash ab12cd34 (dataset v2)
  │           ├── split: splits/v8.dvc hash ef56gh78
  │           └── code: maia-ml-core v1.2.0
  ├── uses model: spine-instance v4 (from MLflow)
  │     └── (same chain)
  ├── uses code: maia-ml-core v1.2.0
  └── uses: segrefine/base:1.0.0
```

**Every link is traceable.** That's what FDA wants.

## FDA 510(k) Traceability Path

When an auditor asks: "Prove that model v9 was trained on approved data."

```
Step 1: Identify deployed model
  → MLflow Registry: spine-semantic-model, stage=Production, version=9

Step 2: Find training run
  → MLflow Run: run_id=abc123
  → Params: lr=0.001, batch_size=8, dataset_version=v2, split=v8
  → Metrics: val_dice_mean=0.891, val_dice_cervical=0.912, ...
  → Artifacts: config.yaml, training_curves.png, model.pth

Step 3: Find exact data
  → MLflow param: dvc_data_hash=ab12cd34
  → Git log in maia-data-registry: find commit with this hash
  → dvc checkout → exact dataset restored

Step 4: Find exact code
  → MLflow tag: git_commit=def456
  → git checkout def456 in maia-spine-training
  → Exact training code + config restored

Step 5: Reproduce
  → dvc repro → identical training run
  → Compare metrics → should match within tolerance
```

**Total time to answer: 5 minutes, not 5 days.**

## Migration Path: Current → Target

Don't do everything at once. Progressive steps:

```
PHASE A: Quick Wins (1-2 weeks)
──────────────────────────────
1. Add DVC to training repo
   - dvc init
   - dvc add data/ (current datasets)
   - dvc remote add azure blob
   
2. Add MLflow to training scripts
   - ~10 lines of code per training script
   - Log params, metrics, model
   - Local file:// tracking first

PHASE B: Structure (2-4 weeks)
──────────────────────────────
3. Create maia-data-registry repo
   - Move .dvc files there
   - Document datasets

4. Extract maia-ml-core package
   - config.py, classes.py, transforms/, dataset.py
   - Add unit tests
   - Both repos depend on it

PHASE C: Automation (4-6 weeks)
──────────────────────────────
5. Define DVC pipeline (dvc.yaml)
6. Set up MLflow server on Azure
7. Update inference repo to pull from MLflow
8. CI/CD integration

PHASE D: Documentation (ongoing)
──────────────────────────────
9. Document lineage for FDA
10. Team training on new workflow
```

## Summary

| Component | Current | Target |
|-----------|---------|--------|
| **Repos** | 2 (training + inference), duplicated code | 4 (data-registry, training, inference, shared) |
| **Data** | On NAS, no versioning | Azure Blob + DVC, fully versioned |
| **Experiments** | TensorBoard only | MLflow (+ TensorBoard kept) |
| **Models** | `.pth` in folders | MLflow Registry with stages |
| **Pipeline** | Manual scripts | DVC pipeline (`dvc repro`) |
| **Shared code** | Copy-paste | `maia-ml-core` Python package |
| **Configs** | Inconsistent naming | Standardized, versioned in Git |
| **CI/CD** | Inference only | Training + inference + shared |
| **Weights in Docker** | Bundled at build | Pulled from MLflow at build |
| **FDA lineage** | Non-existent | Full chain: data → code → model → deployment |

**This blueprint is your roadmap.** Start with Phase A (DVC + MLflow), then progressively build toward the full architecture.

**Next:** Reflection — Q&A, pitfalls, interesting facts, and what to learn next.